# Validation checks (issue #10 follow-up)

Four groups of checks on the final pipeline (42 column XGBoost, gate 0.10 / 0.90):

1. Split check: train, validation and test are disjoint sets of SIMs.
2. Leakage checks: label shuffle, a temporal (no future days) check, a column review, and fit-on-train-only.
3. Error analysis on test at the flag gate (p > 0.90).
4. Alternative models on the same 42 columns.

Nothing here writes a model or an artifact. All refits stay in memory. Thresholds for the alternative
models are chosen on validation only; test is only read. Logged to the MLflow experiment
`simguard-stage5-evaluation`.

In [1]:
import os, sys, json, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import xgboost as xgb
import mlflow
from sklearn.ensemble import IsolationForest, RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (average_precision_score, roc_auc_score, precision_score, recall_score,
                             precision_recall_curve)

warnings.filterwarnings("ignore")
if not os.path.isdir("data") and os.path.isdir(os.path.join("..", "data")):
    os.chdir("..")
sys.path.insert(0, os.getcwd())
ART = Path("artifacts").resolve()
os.environ["ML_ARTIFACT_DIR"] = str(ART)
from src import risk_scoring as rs
from src import feature_engineering as fe

SEED = 42
SIM_START = pd.Timestamp("2025-01-01")
CLEAR, FLAG = rs.GATE["clear_below"], rs.GATE["flag_above"]
CATS = ["ghost_usage", "unauthorized_sharing", "data_reselling", "inactive_billing"]
SPLITS = ("train", "val", "test")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 80)
TIMES = {}

mlflow.set_tracking_uri("sqlite:///mlruns.db")
mlflow.set_experiment("simguard-stage5-evaluation")
if mlflow.active_run():
    mlflow.end_run()
run = mlflow.start_run(run_name="validation_checks")
mlflow.set_tags({"issue": "10", "part": "validation checks", "model": "xgboost_triage.joblib (42 columns)"})

def log_table(df, name):
    mlflow.log_text(df.to_csv(), f"tables/{name}.csv")

model = joblib.load(ART / "xgboost_triage.joblib")
cols42 = json.load(open(ART / "xgboost_feature_columns.json"))
XGB_PARAMS = dict(n_estimators=500, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                  eval_metric="aucpr", early_stopping_rounds=50, tree_method="hist", random_state=SEED, n_jobs=-1)
print("CWD:", os.getcwd(), "| gate", CLEAR, FLAG, "| saved best_iteration", model.best_iteration)

C:\Users\ADMIN\Desktop\projects\simguard\ml-service\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CWD: C:\Users\ADMIN\Desktop\projects\simguard\ml-service | gate 0.1 0.9 | saved best_iteration 56


## 0. Data, labels and saved split files

In [2]:
t0 = time.perf_counter()
daily = pd.read_csv("data/synthetic_cdr_daily_agg.csv", parse_dates=["date"])
episodes = pd.read_csv("data/synthetic_cdr_labels.csv")
episodes["start_dt"] = SIM_START + pd.to_timedelta(episodes["start_day"], unit="D")
episodes["end_dt"] = SIM_START + pd.to_timedelta(episodes["end_day"], unit="D")

Y = {s: pd.read_parquet(f"data/y_{s}.parquet") for s in SPLITS}
X1 = {s: pd.read_parquet(f"data/X_{s}.parquet") for s in SPLITS}      # scaled, before cohort Z
X2 = {s: pd.read_parquet(f"data/X_{s}_v2.parquet") for s in SPLITS}   # training input of xgboost_final.ipynb
for s in SPLITS:
    for X in (X1[s], X2[s]):
        assert (X[["sim_id", "date"]].values == Y[s][["sim_id", "date"]].values).all(), s
split_sims = {s: set(Y[s]["sim_id"].unique()) for s in SPLITS}
labels = pd.concat(Y.values()).sort_values(["sim_id", "date"]).reset_index(drop=True)
TIER = daily.drop_duplicates("sim_id").set_index("sim_id")["tier"]
print({s: X2[s].shape for s in SPLITS})
TIMES["0 data"] = time.perf_counter() - t0

{'train': (63000, 48), 'val': (13500, 48), 'test': (13500, 48)}


## 1. Split check

The split is by SIM: `feature_engineering.ipynb` cell 6 shuffles the sorted SIM ids with
`default_rng(42)` and cuts 700 / 150 / 150. Checked here: the SIM id sets do not overlap, every SIM
is in exactly one split, and every SIM keeps all of its 90 days inside its split (no SIM has rows in
two splits). The split function is also re-run and must give the same sets.

In [3]:
t0 = time.perf_counter()
tr, va, te = split_sims["train"], split_sims["val"], split_sims["test"]
assert not (tr & va) and not (tr & te) and not (va & te), "SIM ids overlap between splits"
all_sims = set(daily["sim_id"].unique())
assert tr | va | te == all_sims and len(tr) + len(va) + len(te) == len(all_sims)
# Split by SIM, not by row: each SIM's rows are all in one split, and all 90 days are there.
rows_per_sim = {s: Y[s].groupby("sim_id").size() for s in SPLITS}
for s in SPLITS:
    assert (rows_per_sim[s] == 90).all(), s
owner = pd.concat([pd.Series(s, index=sorted(split_sims[s])) for s in SPLITS])
assert owner.index.is_unique
ALL_SIMS = np.sort(daily["sim_id"].unique())
def make_split(seed):
    v = ALL_SIMS.copy()
    np.random.default_rng(seed).shuffle(v)
    return {"train": set(v[:700]), "val": set(v[700:850]), "test": set(v[850:])}
assert make_split(42) == split_sims
print("No SIM id is shared between train, validation and test. Every SIM has all 90 days in one split.")
print("Split function with seed 42 reproduces the saved split: True")

rows = []
for s in SPLITS:
    y = Y[s]
    ep = episodes[episodes["sim_id"].isin(split_sims[s])]
    r = {"split": s, "SIMs": len(split_sims[s]), "rows": len(y),
         "anomalous SIMs": ep["sim_id"].nunique(), "anomalous rows": int(y["is_anomalous"].sum()),
         "anomaly rate (rows)": y["is_anomalous"].mean()}
    for c in CATS:
        r[f"{c} rows"] = int((y["misuse_category"] == c).sum())
        r[f"{c} SIMs"] = ep.loc[ep["category"] == c, "sim_id"].nunique()
    for t in fe.TIER_VALUES:
        r[f"tier {t} SIMs"] = int((TIER.loc[sorted(split_sims[s])] == t).sum())
    rows.append(r)
split_tab = pd.DataFrame(rows).set_index("split")
print("Department is not in the data; tier is the only SIM attribute.")
print(split_tab.T.to_string())
log_table(split_tab, "split_counts")
mlflow.log_metric("split_overlap_sims", 0)
TIMES["1 split check"] = time.perf_counter() - t0

No SIM id is shared between train, validation and test. Every SIM has all 90 days in one split.
Split function with seed 42 reproduces the saved split: True
Department is not in the data; tier is the only SIM attribute.
split                             train           val          test
SIMs                         700.000000    150.000000    150.000000
rows                       63000.000000  13500.000000  13500.000000
anomalous SIMs               102.000000     37.000000     23.000000
anomalous rows              1292.000000    524.000000    361.000000
anomaly rate (rows)            0.020508      0.038815      0.026741
ghost_usage rows             447.000000    150.000000     80.000000
ghost_usage SIMs              24.000000     10.000000      8.000000
unauthorized_sharing rows    403.000000    217.000000    235.000000
unauthorized_sharing SIMs     39.000000     18.000000     10.000000
data_reselling rows          293.000000    101.000000     24.000000
data_reselling SIMs           31

## 2a. Label shuffle

The anomaly labels of the training rows are randomly permuted (5 different permutations), so the
labels no longer belong to the rows. XGBoost is then refitted on the same 42 columns with the same
hyperparameters as the final model and a FIXED 57 trees (the final model's tree count), with no early
stopping. If the features or the procedure leaked the label, AP would stay high. Pass rule, fixed
before this run: for all 5 shuffles, validation AP and test AP must each be at most 2 times the base
rate (the share of anomalous rows: about 0.039 on validation and 0.027 on test).

In [4]:
t0 = time.perf_counter()
ytr = Y["train"]["is_anomalous"].values
yva = Y["val"]["is_anomalous"].values
yte = Y["test"]["is_anomalous"].values
spw = (ytr == 0).sum() / (ytr == 1).sum()
N_TREES = int(model.best_iteration) + 1
assert N_TREES == 57
FIXED_PARAMS = {**XGB_PARAMS, "n_estimators": N_TREES, "early_stopping_rounds": None}
base_va, base_te = yva.mean(), yte.mean()
perms = [np.random.default_rng(100 + k).permutation(ytr) for k in range(5)]
expected_kept = ytr.sum() ** 2 / len(ytr)

shuf = []
for k, y_perm in enumerate(perms):
    m = xgb.XGBClassifier(**FIXED_PARAMS, scale_pos_weight=spw)
    m.fit(X2["train"][cols42], y_perm, verbose=False)
    shuf.append({"shuffle": k + 1, "trees": N_TREES,
                 "anomalous labels kept on anomalous rows": int((y_perm & ytr).sum()),
                 "val AP": average_precision_score(yva, m.predict_proba(X2["val"][cols42])[:, 1]),
                 "test AP": average_precision_score(yte, m.predict_proba(X2["test"][cols42])[:, 1])})
shuf = pd.DataFrame(shuf).set_index("shuffle")
shuf["val AP / base rate"] = shuf["val AP"] / base_va
shuf["test AP / base rate"] = shuf["test AP"] / base_te
print("Fixed 57 trees, no early stopping (headline check):")
print(shuf.round(4).to_string())
print(f"base rate: validation {base_va:.4f}, test {base_te:.4f} | anomalous labels kept expected by chance: {expected_kept:.1f}")
print(f"real labels for comparison: validation AP 0.9254, test AP 0.9297")
log_table(shuf, "label_shuffle_fixed_57_trees")
mlflow.log_metric("shuffle_fixed_val_AP_max", shuf["val AP"].max())
mlflow.log_metric("shuffle_fixed_test_AP_max", shuf["test AP"].max())
mlflow.log_metric("val_base_rate", base_va); mlflow.log_metric("test_base_rate", base_te)
assert (shuf["val AP"] <= 2 * base_va).all() and (shuf["test AP"] <= 2 * base_te).all(),     "shuffled labels give AP above 2 times the base rate: look for leakage"
print("PASS: every shuffle is at most 2 times the base rate on validation and test.")
TIMES["2a label shuffle"] = time.perf_counter() - t0

Fixed 57 trees, no early stopping (headline check):
         trees  anomalous labels kept on anomalous rows  val AP  test AP  val AP / base rate  test AP / base rate
shuffle                                                                                                          
1           57                                       23  0.0219   0.0150              0.5631               0.5594
2           57                                       16  0.0207   0.0139              0.5332               0.5210
3           57                                       32  0.0318   0.0183              0.8194               0.6859
4           57                                       23  0.0240   0.0152              0.6178               0.5690
5           57                                       23  0.0266   0.0177              0.6843               0.6606
base rate: validation 0.0388, test 0.0267 | anomalous labels kept expected by chance: 26.5
real labels for comparison: validation AP 0.9254, test AP 0

**Early-stopping version (shown, not asserted).** The same 5 shuffles refitted with early stopping
(patience 50), as `xgboost_final.ipynb` does. Early stopping selects the tree count using the real
validation labels, so with shuffled training labels it can keep a one-tree model whose direction
happens to match the validation labels. That is what happens in shuffle 4 (validation AP 0.179, test
AP 0.146), although only 23 anomalous labels landed on anomalous rows versus 26.5 expected by chance.
Its validation AP falls with every added tree, which is why the fixed tree count above is the fair
check.

In [5]:
t0 = time.perf_counter()
shuf_es = []
for k, y_perm in enumerate(perms):
    m = xgb.XGBClassifier(**XGB_PARAMS, scale_pos_weight=spw)
    m.fit(X2["train"][cols42], y_perm, eval_set=[(X2["val"][cols42], yva)], verbose=False)
    curve = m.evals_result()["validation_0"]["aucpr"]
    shuf_es.append({"shuffle": k + 1, "trees kept": int(m.best_iteration) + 1,
                    "anomalous labels kept on anomalous rows": int((y_perm & ytr).sum()),
                    "val AP": average_precision_score(yva, m.predict_proba(X2["val"][cols42])[:, 1]),
                    "test AP": average_precision_score(yte, m.predict_proba(X2["test"][cols42])[:, 1]),
                    "val aucpr after 1 / 5 / 51 trees": f"{curve[0]:.3f} / {curve[4]:.3f} / {curve[50]:.3f}"})
shuf_es = pd.DataFrame(shuf_es).set_index("shuffle")
print("Early stopping, patience 50 (not asserted):")
print(shuf_es.round(4).to_string())
log_table(shuf_es, "label_shuffle_early_stopping")
mlflow.log_metric("shuffle_es_val_AP_max", shuf_es["val AP"].max())
TIMES["2a label shuffle (early stopping)"] = time.perf_counter() - t0

Early stopping, patience 50 (not asserted):
         trees kept  anomalous labels kept on anomalous rows  val AP  test AP val aucpr after 1 / 5 / 51 trees
shuffle                                                                                                       
1                 3                                       23  0.0341   0.0329            0.029 / 0.028 / 0.022
2                 1                                       16  0.0420   0.0444            0.041 / 0.021 / 0.021
3                 4                                       32  0.0649   0.0475            0.050 / 0.063 / 0.034
4                 1                                       23  0.1787   0.1458            0.192 / 0.051 / 0.024
5                 4                                       23  0.0610   0.0453            0.051 / 0.037 / 0.029


## 2b. Temporal check (no feature uses future days)

30 test SIMs (15 with an episode, 15 without) and 3 random dates each. For each (SIM, date), the
daily input is cut to that SIM's days up to and including the scored date, and `src.risk_scoring.score`
is run on it. The probability must be identical to the run on the full 90 days. Any feature that
looked at later days (for example a centred window or a whole-cycle total) would change.

In [6]:
t0 = time.perf_counter()
rng = np.random.default_rng(7)
test_anom = sorted(set(episodes["sim_id"]) & te)
test_norm = sorted(te - set(episodes["sim_id"]))
sims30 = list(rng.choice(test_anom, 15, replace=False)) + list(rng.choice(test_norm, 15, replace=False))
d_test = daily[daily["sim_id"].isin(sims30)]
full = rs.score(d_test, artifact_dir=ART).set_index(["sim_id", "date"])
trows = []
for sim in sims30:
    days = np.sort(rng.choice(np.arange(90), 3, replace=False))
    for dd in days:
        day = SIM_START + pd.Timedelta(days=int(dd))
        cut = daily[(daily["sim_id"] == sim) & (daily["date"] <= day)]
        out = rs.score(cut, as_of_dates=day, artifact_dir=ART)
        assert len(out) == 1
        p_full = full.loc[(sim, day), "probability"]
        trows.append({"sim_id": sim, "day": int(dd), "rows given": len(cut), "p full": p_full,
                      "p cut": out["probability"].iloc[0], "abs diff": abs(out["probability"].iloc[0] - p_full),
                      "same bucket": out["bucket"].iloc[0] == full.loc[(sim, day), "bucket"],
                      "anomalous": int(labels.set_index(["sim_id", "date"]).loc[(sim, day), "is_anomalous"])})
temporal = pd.DataFrame(trows)
print(f"{len(temporal)} (SIM, date) pairs | days used: min {temporal['day'].min()}, max {temporal['day'].max()} | "
      f"anomalous pairs {temporal['anomalous'].sum()}")
print(f"max |p cut - p full| = {temporal['abs diff'].max():.3e} | same bucket: {temporal['same bucket'].mean():.0%}")
print(temporal.sort_values("abs diff", ascending=False).head(5).round(6).to_string(index=False))
log_table(temporal, "temporal_check")
mlflow.log_metric("temporal_max_abs_diff", float(temporal["abs diff"].max()))
assert temporal["abs diff"].max() == 0.0, "a feature changes when future days are removed"
TIMES["2b temporal"] = time.perf_counter() - t0

90 (SIM, date) pairs | days used: min 0, max 88 | anomalous pairs 5
max |p cut - p full| = 0.000e+00 | same bucket: 100%
 sim_id  day  rows given   p full    p cut  abs diff  same bucket  anomalous
    117   23          24 0.078926 0.078926       0.0         True          0
    117   38          39 0.106175 0.106175       0.0         True          0
    117   45          46 0.087704 0.087704       0.0         True          0
    508    1           2 0.120849 0.120849       0.0         True          0
    508   21          22 0.087659 0.087659       0.0         True          0


## 2c. Column check

No column of the 42 is read from the label file: all come from the daily usage table
(`build_features`), the Isolation Forest score, or the cohort Z of those. The names are checked below
for label words. Then, because the generator creates each anomaly by changing specific raw fields
(`synthetic_cdr_generator.ipynb` cell 16), each column is scored alone: its best single-column AUC
against each category on the training rows (using the column or its negative). A column that alone
separates a category almost perfectly is a column that mirrors how the generator built that category.

In [7]:
t0 = time.perf_counter()
bad_words = ("anomal", "misuse", "category", "label", "episode", "ghost", "resell", "sharing", "inactive", "target")
named = [c for c in cols42 if any(w in c.lower() for w in bad_words)]
print("columns with a label word in the name:", named or "none")
# Reviewed: the word "anomaly" in these names is not the label.
REVIEWED = {"has_geo_anomaly": "geo_flag_count > 0 from the session velocity check in the daily table, no label input",
            "cohort_z_iso_anomaly_score": "cohort Z of the unsupervised Isolation Forest score, fitted without labels"}
for c in named:
    print(f"  {c}: {REVIEWED.get(c, 'NOT REVIEWED')}")
assert set(named) <= set(REVIEWED), f"unreviewed column names: {set(named) - set(REVIEWED)}"
# Every model column is a build_features column or a cohort Z of one; none is read from the label file.
assert set(cols42) <= set(fe.FEATURE_COLUMNS) | set(rs.COHORT_Z_COLUMNS)

# Unscaled feature values for the training rows (for readable numbers).
FEATS = fe.build_features(daily, ART).reset_index(drop=True)
FEATS = FEATS.merge(labels, on=["sim_id", "date"], validate="one_to_one")
ftr = FEATS[FEATS["sim_id"].isin(tr)].reset_index(drop=True)
X2tr = X2["train"].reset_index(drop=True)
assert (ftr[["sim_id", "date"]].values == X2tr[["sim_id", "date"]].values).all()
normal = (ftr["misuse_category"] == "normal").values
auc_rows = []
for c in cols42:
    v = X2tr[c].values
    r = {"column": c}
    for cat in CATS:
        m = normal | (ftr["misuse_category"] == cat).values
        a = roc_auc_score((ftr["misuse_category"].values[m] == cat).astype(int), v[m])
        r[cat] = max(a, 1 - a)
    auc_rows.append(r)
auc_tab = pd.DataFrame(auc_rows).set_index("column")
auc_tab["best"] = auc_tab.max(axis=1)
auc_tab["best category"] = auc_tab[CATS].idxmax(axis=1)
print(auc_tab.sort_values("best", ascending=False).head(15).round(3).to_string())

# How cleanly the generator fingerprints show up in the raw columns (training rows).
fp = pd.DataFrame({
    "billed > used by more than 1 MB": (ftr["billed_mb"] - ftr["total_mb"] > 1),
    "total_mb below 6 MB": ftr["total_mb"] < 6,
    "total_mb exactly 0": ftr["total_mb"] == 0,
    "hotspot_sessions >= 2": ftr["hotspot_sessions"] >= 2,
}).groupby(ftr["misuse_category"]).mean()
print(); print("Share of training rows with each pattern, by category:")
print(fp.round(3).to_string())
log_table(auc_tab, "single_column_auc"); log_table(fp, "generator_fingerprints")
TIMES["2c columns"] = time.perf_counter() - t0

columns with a label word in the name: ['has_geo_anomaly', 'cohort_z_iso_anomaly_score']
  has_geo_anomaly: geo_flag_count > 0 from the session velocity check in the daily table, no label input
  cohort_z_iso_anomaly_score: cohort Z of the unsupervised Isolation Forest score, fitted without labels


                            ghost_usage  unauthorized_sharing  data_reselling  inactive_billing   best         best category
column                                                                                                                      
upload_ratio                      0.522                 0.520           0.527             0.974  0.974      inactive_billing
total_upload_mb                   0.974                 0.783           0.641             0.960  0.974           ghost_usage
cohort_z_total_mb                 0.973                 0.854           0.670             0.958  0.973           ghost_usage
billed_mb                         0.972                 0.803           0.655             0.948  0.972           ghost_usage
total_mb                          0.972                 0.803           0.655             0.954  0.972           ghost_usage
total_download_mb                 0.971                 0.797           0.656             0.953  0.971           ghost_usage


## 2d. Fit on train only

How it is checked: each fitted object is refitted here on the 700 training SIMs only and compared with
the saved artifact, and also refitted on all 1,000 SIMs to show that the comparison can tell the two
apart. The training notebooks were also read: `feature_engineering.ipynb` cell 7 fits the scaler on
`df_train`; `isolation_forest.ipynb` cell 8 fits on `X_train`; `cohort_zscore.ipynb` cell 7 computes the
stats from `X_train_raw`; `xgboost_final.ipynb` fits on `X_train_v2` with `X_val_v2` only as the early
stopping set. The SIM ids inside each of those files are checked against the split.

In [8]:
t0 = time.perf_counter()
fit_rows = []
for f in ("X_train", "X_train_v2", "y_train"):
    ids = set(pd.read_parquet(f"data/{f}.parquet", columns=["sim_id"])["sim_id"])
    fit_rows.append({"object": f"{f}.parquet", "fitted on": "file", "SIMs": len(ids),
                     "SIM ids equal to the train split": ids == tr, "max diff to saved": np.nan})

# Scaler: refit on the unscaled training rows.
saved_scaler = joblib.load(ART / "feature_scaler.joblib")
Xs = FEATS[list(fe.FEATURE_COLUMNS)].copy()
for c in fe.UNSCALED_CLIPPED_COLUMNS:
    Xs[c] = Xs[c].clip(-fe.ZSCORE_CLIP, fe.ZSCORE_CLIP)
in_tr = FEATS["sim_id"].isin(tr).values
def scaler_diff(mask):
    sc = RobustScaler().fit(Xs.loc[mask, list(fe.SCALED_COLUMNS)])
    return max(np.abs(sc.center_ - saved_scaler.center_).max(), np.abs(sc.scale_ - saved_scaler.scale_).max())
fit_rows.append({"object": "feature_scaler.joblib", "fitted on": "700 train SIMs", "SIMs": 700,
                 "max diff to saved": scaler_diff(in_tr)})
fit_rows.append({"object": "feature_scaler.joblib", "fitted on": "all 1,000 SIMs", "SIMs": 1000,
                 "max diff to saved": scaler_diff(np.ones(len(Xs), bool))})

# Isolation Forest: refit on X_train.parquet, compare decision_function on test rows.
saved_iso = joblib.load(ART / "isolation_forest.joblib")
FC = list(fe.FEATURE_COLUMNS)
X1all = pd.concat([X1[s] for s in SPLITS], ignore_index=True)
def iso_diff(Xfit):
    iso = IsolationForest(n_estimators=200, contamination="auto", random_state=SEED, n_jobs=-1).fit(Xfit[FC])
    return np.abs(iso.decision_function(X1["test"][FC]) - saved_iso.decision_function(X1["test"][FC])).max()
fit_rows.append({"object": "isolation_forest.joblib", "fitted on": "700 train SIMs", "SIMs": 700,
                 "max diff to saved": iso_diff(X1["train"])})
fit_rows.append({"object": "isolation_forest.joblib", "fitted on": "all 1,000 SIMs", "SIMs": 1000,
                 "max diff to saved": iso_diff(X1all)})

# Cohort stats: per tier median and IQR of the four source columns.
saved_stats = json.load(open(ART / "cohort_stats.json"))
def cohort_diff(Xpart):
    w = Xpart.copy()
    w["iso_anomaly_score"] = -saved_iso.decision_function(w[FC])
    tier = w[list(rs.TIER_COLUMNS)].idxmax(axis=1).str.replace("tier_", "", regex=False)
    d = 0.0
    for col in rs.COHORT_SOURCE_COLUMNS:
        g = w[col].groupby(tier)
        med, iqr = g.median(), g.quantile(0.75) - g.quantile(0.25)
        for t in saved_stats[col]["median"]:
            d = max(d, abs(med[t] - saved_stats[col]["median"][t]), abs(iqr[t] - saved_stats[col]["iqr"][t]))
    return d
fit_rows.append({"object": "cohort_stats.json", "fitted on": "700 train SIMs", "SIMs": 700,
                 "max diff to saved": cohort_diff(X1["train"])})
fit_rows.append({"object": "cohort_stats.json", "fitted on": "all 1,000 SIMs", "SIMs": 1000,
                 "max diff to saved": cohort_diff(X1all)})

# XGBoost: refit on X_train_v2 (early stopping on X_val_v2), compare test probabilities.
p_saved_te = model.predict_proba(X2["test"][cols42])[:, 1]
m = xgb.XGBClassifier(**XGB_PARAMS, scale_pos_weight=spw)
m.fit(X2["train"][cols42], ytr, eval_set=[(X2["val"][cols42], yva)], verbose=False)
fit_rows.append({"object": "xgboost_triage.joblib", "fitted on": "700 train SIMs (early stopping on 150 val SIMs)",
                 "SIMs": 700, "max diff to saved": np.abs(m.predict_proba(X2["test"][cols42])[:, 1] - p_saved_te).max()})
Xall2 = pd.concat([X2[s] for s in SPLITS], ignore_index=True)
yall = np.concatenate([Y[s]["is_anomalous"].values for s in SPLITS])
m_all = xgb.XGBClassifier(**{**XGB_PARAMS, "early_stopping_rounds": None, "n_estimators": int(model.best_iteration) + 1},
                          scale_pos_weight=(yall == 0).sum() / (yall == 1).sum())
m_all.fit(Xall2[cols42], yall, verbose=False)
fit_rows.append({"object": "xgboost_triage.joblib", "fitted on": "all 1,000 SIMs (same tree count)", "SIMs": 1000,
                 "max diff to saved": np.abs(m_all.predict_proba(X2["test"][cols42])[:, 1] - p_saved_te).max()})
fit_tab = pd.DataFrame(fit_rows)
print(fit_tab.to_string(index=False))
log_table(fit_tab, "fit_on_train_only")
assert all(fit_tab.loc[fit_tab["fitted on"] == "file", "SIM ids equal to the train split"])
on_train = fit_tab["fitted on"].str.startswith("700")
assert (fit_tab.loc[on_train, "max diff to saved"] < 1e-6).all(), "an artifact is not reproduced from train SIMs"
assert (fit_tab.loc[fit_tab["fitted on"].str.startswith("all"), "max diff to saved"] > 1e-4).all()
TIMES["2d fit on train"] = time.perf_counter() - t0

                 object                                       fitted on  SIMs SIM ids equal to the train split  max diff to saved
        X_train.parquet                                            file   700                             True                NaN
     X_train_v2.parquet                                            file   700                             True                NaN
        y_train.parquet                                            file   700                             True                NaN
  feature_scaler.joblib                                  700 train SIMs   700                              NaN       7.105427e-15
  feature_scaler.joblib                                  all 1,000 SIMs  1000                              NaN       1.431648e+01
isolation_forest.joblib                                  700 train SIMs   700                              NaN       0.000000e+00
isolation_forest.joblib                                  all 1,000 SIMs  1000             

## 3. Error analysis on test at the flag gate

Scores come from `src.risk_scoring.score` on the test SIMs' raw daily rows (identical to the notebook
probabilities, see `end_to_end_evaluation.ipynb` part 1). SHAP values are the model's own
`pred_contribs` limited to the trees in use. "Cohort median" is the median of the column over the
normal training rows of the same tier. Values are unscaled.

In [9]:
t0 = time.perf_counter()
d_te = daily[daily["sim_id"].isin(te)]
feats_te = fe.build_features(d_te, ART).reset_index(drop=True)
scaled_te = fe.scale_features(feats_te, ART)
out_te, det = rs.score_features(scaled_te, ART, return_details=True)
out_te = out_te.merge(Y["test"], on=["sim_id", "date"], validate="one_to_one")
assert (out_te[["sim_id", "date"]].values == feats_te[["sim_id", "date"]].values).all()
assert np.abs(out_te["probability"].values - p_saved_te).max() < 1e-4
contrib = pd.DataFrame(det["contributions"], columns=cols42)
out_te["tier"] = out_te["sim_id"].map(TIER)
ep_of = {}
for e in episodes.itertuples(index=False):
    for k in range(e.start_day, e.end_day + 1):
        ep_of[(e.sim_id, SIM_START + pd.Timedelta(days=k))] = (e.category, k - e.start_day, e.duration_days)
out_te["episode_day"] = [ep_of.get((s, d), (None, np.nan, np.nan))[1] for s, d in zip(out_te["sim_id"], out_te["date"])]
out_te["episode_len"] = [ep_of.get((s, d), (None, np.nan, np.nan))[2] for s, d in zip(out_te["sim_id"], out_te["date"])]

# Reference: per tier median of the normal training rows (unscaled).
SHOW = ["total_mb", "hotspot_sessions", "offpeak_ratio", "upload_ratio", "public_apn_ratio",
        "unique_towers", "data_session_count", "billing_ratio", "vol_zscore_7d"]
ref = ftr[ftr["misuse_category"] == "normal"].assign(tier=lambda d: d["sim_id"].map(TIER)).groupby("tier")[SHOW].median()
own_median = feats_te.assign(lab=out_te["misuse_category"].values).query("lab == 'normal'").groupby("sim_id")["total_mb"].median()

def top3(i):
    c = contrib.iloc[i]
    o = c.abs().sort_values(ascending=False).index[:3]
    return [f"{f} ({c[f]:+.2f})" for f in o]

def describe(i):
    f, t = feats_te.iloc[i], out_te["tier"].iloc[i]
    return {s: f"{f[s]:.2f} vs {ref.loc[t, s]:.2f}" for s in ("total_mb", "hotspot_sessions", "offpeak_ratio", "upload_ratio", "unique_towers")}
TIMES["3 scoring"] = time.perf_counter() - t0

### 3a. False positives

In [10]:
t0 = time.perf_counter()
flag = out_te["bucket"] == "flag"
fp_idx = np.where(flag & (out_te["is_anomalous"] == 0))[0]
anom_sims = set(episodes["sim_id"])
fp_rows = []
for i in fp_idx:
    s, d = out_te["sim_id"].iloc[i], out_te["date"].iloc[i]
    eps = episodes[episodes["sim_id"] == s]
    if len(eps):
        gap = min(min(abs((d - e.start_dt).days), abs((d - e.end_dt).days)) for e in eps.itertuples())
        where = f"anomalous SIM ({','.join(eps['category'])}), {gap} day(s) from an episode"
    else:
        where = "normal SIM"
    r = {"sim_id": s, "date": d.date(), "tier": out_te["tier"].iloc[i], "p": round(out_te["probability"].iloc[i], 3),
         "SIM type": where, "top 3 SHAP": "; ".join(top3(i)),
         "total_mb / own normal median": round(feats_te["total_mb"].iloc[i] / own_median.get(s, np.nan), 2)}
    r.update({f"{k} (row vs cohort)": v for k, v in describe(i).items()})
    fp_rows.append(r)
fp_tab = pd.DataFrame(fp_rows)
print(f"False positive rows (normal rows flagged): {len(fp_tab)} of {int(flag.sum())} flagged rows")
print(fp_tab.T.to_string())

# SIM-level false alarms: normal SIMs (no episode) with at least one flag day.
fa_sims = sorted(set(out_te.loc[flag, "sim_id"]) - anom_sims)
fa_rows = []
for s in fa_sims:
    o = out_te[out_te["sim_id"] == s]
    i = o["probability"].idxmax()
    days_flagged = o.loc[o["bucket"] == "flag", "date"].dt.date.tolist()
    r = {"sim_id": s, "tier": TIER[s], "flag days": len(days_flagged), "dates": days_flagged,
         "max p": round(o["probability"].max(), 3), "escalated days": int((o["bucket"] == "escalated").sum()),
         "top 3 SHAP on max day": "; ".join(top3(i)),
         "total_mb on max day / own median": round(feats_te["total_mb"].iloc[i] / own_median[s], 2),
         "own median total_mb vs cohort": f"{own_median[s]:.0f} vs {ref.loc[TIER[s], 'total_mb']:.0f}"}
    r.update({f"{k} (max day vs cohort)": v for k, v in describe(i).items()})
    fa_rows.append(r)
fa_tab = pd.DataFrame(fa_rows)
print(); print(f"SIM-level false alarms: {len(fa_tab)} normal SIMs with at least one flag day")
print(fa_tab.T.to_string())
log_table(fp_tab, "false_positive_rows"); log_table(fa_tab, "false_alarm_sims")
TIMES["3a false positives"] = time.perf_counter() - t0

False positive rows (normal rows flagged): 5 of 304 flagged rows
                                                                                                   0                                                                                            1                                                                           2                                                                   3                                                                                            4
sim_id                                                                                           117                                                                                          195                                                                         528                                                                 641                                                                                          918
date                                                                     

### 3b. False negatives (anomalous rows not flagged)

In [11]:
t0 = time.perf_counter()
anom = out_te["is_anomalous"] == 1
missed, caught = anom & ~flag, anom & flag
fn_bucket = (out_te[missed].groupby(["misuse_category", "bucket"]).size().unstack(fill_value=0)
             .reindex(CATS).fillna(0).astype(int))
fn_bucket["missed total"] = fn_bucket.sum(axis=1)
fn_bucket["anomalous rows"] = out_te[anom].groupby("misuse_category").size().reindex(CATS)
fn_bucket["missed SIMs"] = out_te[missed].groupby("misuse_category")["sim_id"].nunique().reindex(CATS).fillna(0).astype(int)
print(fn_bucket.to_string()); print()

shap_rows = []
for c in CATS:
    for name, m in (("missed", missed), ("caught", caught)):
        mm = (m & (out_te["misuse_category"] == c)).values
        if not mm.any():
            continue
        mean_abs = contrib[mm].abs().mean().sort_values(ascending=False)
        mean_sgn = contrib[mm].mean()
        shap_rows.append({"category": c, "group": name, "rows": int(mm.sum()),
                          "top 3 by mean abs SHAP (mean signed)": "; ".join(f"{f} ({mean_sgn[f]:+.2f})" for f in mean_abs.index[:3]),
                          "mean margin (log-odds)": det["margin"][mm].mean()})
shap_tab = pd.DataFrame(shap_rows)
print(shap_tab.to_string(index=False)); print()

# When in the episode the missed rows fall, and how strong the change is.
ft = feats_te.assign(own=feats_te["sim_id"].map(own_median))
ft["vol_ratio"] = ft["total_mb"] / ft["own"]
timing_rows = []
for c in CATS:
    for name, m in (("missed", missed), ("caught", caught)):
        mm = (m & (out_te["misuse_category"] == c)).values
        if not mm.any():
            continue
        ed = out_te.loc[mm, "episode_day"]
        timing_rows.append({"category": c, "group": name, "rows": int(mm.sum()),
                            "median episode day (0 = first)": ed.median(),
                            "share on day 0": (ed == 0).mean(), "share on days 0-1": (ed <= 1).mean(),
                            "median total_mb / own normal median": ft.loc[mm, "vol_ratio"].median(),
                            "median hotspot_sessions": ft.loc[mm, "hotspot_sessions"].median(),
                            "median billing_ratio": ft.loc[mm, "billing_ratio"].median(),
                            "median p": out_te.loc[mm, "probability"].median()})
timing_tab = pd.DataFrame(timing_rows)
print(timing_tab.round(3).to_string(index=False))
# data_reselling: a burst (+500 to 3,000 MB) happens on only about 30% of episode days.
rs_mask = (out_te["misuse_category"] == "data_reselling").values
burst = (ft["total_mb"] - ft["own"]) > 400
print(); print("data_reselling rows with a burst (total_mb more than 400 MB above the SIM's normal median):")
print(pd.crosstab(np.where(flag.values[rs_mask], "flagged", "not flagged"), np.where(burst.values[rs_mask], "burst", "no burst")))
log_table(fn_bucket, "false_negatives_by_bucket"); log_table(shap_tab, "false_negative_shap"); log_table(timing_tab, "false_negative_timing")
TIMES["3b false negatives"] = time.perf_counter() - t0

bucket                clear  escalated  missed total  anomalous rows  missed SIMs
misuse_category                                                                  
ghost_usage               3          5             8              80            8
unauthorized_sharing      5         38            43             235           10
data_reselling            3          5             8              24            4
inactive_billing          2          1             3              22            3

            category  group  rows                                                      top 3 by mean abs SHAP (mean signed)  mean margin (log-odds)
         ghost_usage missed     8        peak_mb (-0.93); cohort_z_iso_anomaly_score (-0.27); hotspot_session_ratio (-0.26)               -2.145734
         ghost_usage caught    72                            peak_mb (+2.71); bytes_per_session (+0.50); offpeak_mb (+0.43)                3.489419
unauthorized_sharing missed    43                    hotspot_su

**The first labelled day of each episode.** In `synthetic_cdr_generator.ipynb` cell 16, the day a SIM
enters an episode is recorded as `start_day`, but that day's usage is still drawn as normal; the
volume transform starts the next day. The end day is transformed. The check below measures, over all
175 episodes, how often the category signature is present at each offset from the start and end day
(ghost_usage: total_mb within 1 MB of 5; inactive_billing: total_mb is 0; unauthorized_sharing: at
least one hotspot session; data_reselling has no fixed daily signature and is left out).

In [12]:
t0 = time.perf_counter()
dd = daily.assign(day=(daily["date"] - SIM_START).dt.days)[["sim_id", "day", "total_mb", "hotspot_sessions"]]
SIG = {"ghost_usage": lambda r: (r["total_mb"] - 5).abs() < 1, "inactive_billing": lambda r: r["total_mb"] == 0,
       "unauthorized_sharing": lambda r: r["hotspot_sessions"] >= 1}
sig_rows = []
for c, f in SIG.items():
    g = episodes[episodes["category"] == c]
    for anchor in ("start_day", "end_day"):
        for off in (-1, 0, 1, 2):
            mm = g.assign(day=g[anchor] + off).merge(dd, on=["sim_id", "day"])
            sig_rows.append({"category": c, "anchor": anchor.replace("_day", ""), "offset": off, "share with signature": f(mm).mean()})
sig_tab = pd.DataFrame(sig_rows).pivot_table(index=["category", "anchor"], columns="offset", values="share with signature")
print("Share of episode rows showing the category signature, by day offset from the episode start and end:")
print(sig_tab.round(2).to_string())
assert (sig_tab.xs("start", level="anchor")[0] < 0.05).all() and (sig_tab.xs("start", level="anchor")[1] > 0.9).all()

day0 = anom & (out_te["episode_day"] == 0)
print()
print(f"test: {int(day0.sum())} anomalous rows are episode day 0 (on {out_te.loc[day0, 'sim_id'].nunique()} SIMs); "
      f"flagged {int((day0 & flag).sum())}, escalated {int((day0 & (out_te['bucket'] == 'escalated')).sum())}, cleared {int((day0 & (out_te['bucket'] == 'clear')).sum())}")
print(f"missed rows that are episode day 0: {int((day0 & missed).sum())} of {int(missed.sum())}")
rec_all = (anom & flag).sum() / anom.sum()
rec_no0 = (anom & flag & ~day0).sum() / (anom & ~day0).sum()
print(f"flag recall on test: {rec_all:.3f} on all anomalous rows, {rec_no0:.3f} without episode day 0 (supplementary, the headline is unchanged)")
per_cat0 = out_te[missed].assign(day0=lambda d: d["episode_day"] == 0).groupby("misuse_category")["day0"].sum().reindex(CATS)
print("missed day-0 rows per category:", per_cat0.to_dict())
log_table(sig_tab, "episode_signature_by_offset")
mlflow.log_metric("test_flag_recall_without_episode_day0", rec_no0)
mlflow.log_metric("test_missed_rows_episode_day0", int((day0 & missed).sum()))
TIMES["3b episode day 0"] = time.perf_counter() - t0

Share of episode rows showing the category signature, by day offset from the episode start and end:
offset                         -1     0     1     2
category             anchor                        
ghost_usage          end     0.95  1.00  0.00  0.00
                     start   0.00  0.00  1.00  0.95
inactive_billing     end     1.00  1.00  0.00  0.00
                     start   0.00  0.00  1.00  1.00
unauthorized_sharing end     0.78  0.96  0.14  0.09
                     start   0.13  0.04  0.96  0.82

test: 25 anomalous rows are episode day 0 (on 23 SIMs); flagged 0, escalated 12, cleared 13
missed rows that are episode day 0: 25 of 62
flag recall on test: 0.828 on all anomalous rows, 0.890 without episode day 0 (supplementary, the headline is unchanged)
missed day-0 rows per category: {'ghost_usage': 8, 'unauthorized_sharing': 10, 'data_reselling': 4, 'inactive_billing': 3}


### 3c. Findings

1. **The first labelled day of every episode cannot be detected, by construction.** Every
   episode's first labelled day still carries normal usage in the generator (the signature appears on
   0% of start days and on 96 to 100% of the next days). On test, all 25 of these rows were missed, and
   they make up 25 of the 62 missed rows: every missed ghost_usage and inactive_billing row, 4 of 8
   missed data_reselling rows and 10 of 43 missed unauthorized_sharing rows. Without them, test flag
   recall is 0.890 instead of 0.828. They are also all 13 of the anomalous
   rows that the gate cleared. This is a property of the synthetic labels, not of the model.
2. **The remaining misses are weak versions of the pattern.** Missed unauthorized_sharing rows (38 of 43 are
   escalated, not cleared) have a median of 1 hotspot session and 1.5 times the
   SIM's normal volume, against 3 sessions and 2.5 times on caught rows, because the generator marks
   each session as hotspot only with probability 0.65. data_reselling adds a burst on only about 30% of
   days: all 8 burst days were flagged, and every missed reselling row is a day without a burst.
3. **False alarms are normal days that by chance look like misuse.** The 5 false positive rows (2 SIMs
   with no episode, 3 days on SIMs that had an episode at another time) fall into two patterns: no
   peak-hour traffic at all (offpeak_ratio 1.0, peak_mb is the top reason; 3 rows, the
   data_reselling pattern), or
   2 to 3 hotspot sessions on one day (2 rows, the sharing pattern; normal sessions are hotspot with
   probability 0.03). Each false alarm SIM was flagged on a single day only.

## 4. Alternative models on the same 42 columns

Same training rows (`X_train_v2`, 700 SIMs), same 42 columns, validation used for every choice:

- Logistic regression: standardised inside a pipeline fitted on train, `class_weight="balanced"`, C chosen from 0.01, 0.1, 1, 10 by validation AP.
- Random forest: 300 trees, `class_weight="balanced_subsample"`, `min_samples_leaf` chosen from 1, 5, 20 by validation AP.
- HistGradientBoosting: learning rate 0.05, max depth 6, `class_weight="balanced"`, up to 500 iterations; the number of iterations is chosen as the best validation AP over the stages (the same idea as early stopping, without a random row split inside train).
- Isolation Forest alone (the saved model, no labels) and the final XGBoost (saved model, early stopping patience 50 on validation) as references.

Each model's flag threshold is the lowest validation score with validation precision of at least 0.95
(the README retraining rule). That threshold is then applied to test unchanged. Test is not used for
any choice. For XGBoost the production gate (0.90) is shown as well.

In [13]:
t0 = time.perf_counter()
Xtr, Xva, Xte = (X2[s][cols42] for s in SPLITS)
fitted, picks = {}, {}

best = None
for C in (0.01, 0.1, 1, 10):
    lr = make_pipeline(StandardScaler(), LogisticRegression(C=C, class_weight="balanced", max_iter=5000))
    lr.fit(Xtr, ytr)
    ap = average_precision_score(yva, lr.predict_proba(Xva)[:, 1])
    if best is None or ap > best[0]:
        best = (ap, C, lr)
picks["Logistic regression"] = f"C={best[1]}"; fitted["Logistic regression"] = best[2]
print("logistic regression:", picks["Logistic regression"], f"val AP {best[0]:.4f}")

best = None
for leaf in (1, 5, 20):
    rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=leaf, class_weight="balanced_subsample",
                                random_state=SEED, n_jobs=-1).fit(Xtr, ytr)
    ap = average_precision_score(yva, rf.predict_proba(Xva)[:, 1])
    if best is None or ap > best[0]:
        best = (ap, leaf, rf)
picks["Random forest"] = f"min_samples_leaf={best[1]}"; fitted["Random forest"] = best[2]
print("random forest:", picks["Random forest"], f"val AP {best[0]:.4f}")

hgb = HistGradientBoostingClassifier(learning_rate=0.05, max_depth=6, max_iter=500, early_stopping=False,
                                     class_weight="balanced", random_state=SEED).fit(Xtr, ytr)
stage_ap = [average_precision_score(yva, p[:, 1]) for p in hgb.staged_predict_proba(Xva)]
n_best = int(np.argmax(stage_ap)) + 1
hgb_best = HistGradientBoostingClassifier(learning_rate=0.05, max_depth=6, max_iter=n_best, early_stopping=False,
                                          class_weight="balanced", random_state=SEED).fit(Xtr, ytr)
picks["HistGradientBoosting"] = f"{n_best} iterations"; fitted["HistGradientBoosting"] = hgb_best
print("hist gradient boosting:", picks["HistGradientBoosting"], f"val AP {max(stage_ap):.4f}")

iso_saved = joblib.load(ART / "isolation_forest.joblib")
scores = {name: {s: m.predict_proba(X)[:, 1] for s, X in (("val", Xva), ("test", Xte))} for name, m in fitted.items()}
scores["Isolation Forest alone"] = {s: -iso_saved.decision_function(X1[s][FC]) for s in ("val", "test")}
scores["XGBoost (final)"] = {s: model.predict_proba(X)[:, 1] for s, X in (("val", Xva), ("test", Xte))}
picks["Isolation Forest alone"] = "saved model, no labels"; picks["XGBoost (final)"] = f"saved, {model.best_iteration + 1} trees"
TIMES["4 fit alternatives"] = time.perf_counter() - t0

logistic regression: C=0.01 val AP 0.8083


random forest: min_samples_leaf=5 val AP 0.9293


hist gradient boosting: 490 iterations val AP 0.9225


In [14]:
t0 = time.perf_counter()
ys = {"val": yva, "test": yte}
def recall_at_fpr(y, s, fpr=0.05):
    thr = np.quantile(s[y == 0], 1 - fpr)
    return (s[y == 1] > thr).mean()
def thr_at_precision(y, s, target=0.95):
    prec, rec, thr = precision_recall_curve(y, s)
    ok = np.where(prec[:-1] >= target)[0]
    return thr[ok].min() if len(ok) else np.nan

alt_rows = []
order = ["Isolation Forest alone", "Logistic regression", "Random forest", "HistGradientBoosting", "XGBoost (final)"]
for name in order:
    t = thr_at_precision(yva, scores[name]["val"])
    r = {"model": name, "choice made on validation": picks[name], "threshold (from val)": t}
    for s in ("val", "test"):
        y, sc = ys[s], scores[name][s]
        r[f"{s} AP"] = average_precision_score(y, sc)
        r[f"{s} ROC AUC"] = roc_auc_score(y, sc)
        r[f"{s} recall at 5% FPR"] = recall_at_fpr(y, sc)
        pred = sc >= t if not np.isnan(t) else np.zeros(len(y), bool)
        r[f"{s} flag precision"] = precision_score(y, pred, zero_division=0) if pred.any() else np.nan
        r[f"{s} flag recall"] = recall_score(y, pred)
        r[f"{s} flagged rows"] = int(pred.sum())
    alt_rows.append(r)
# The production gate for reference.
r = {"model": "XGBoost (final), production gate p > 0.90", "choice made on validation": "fixed 0.90", "threshold (from val)": FLAG}
for s in ("val", "test"):
    y, sc = ys[s], scores["XGBoost (final)"][s]
    pred = sc > FLAG
    r.update({f"{s} AP": np.nan, f"{s} ROC AUC": np.nan, f"{s} recall at 5% FPR": np.nan,
              f"{s} flag precision": precision_score(y, pred), f"{s} flag recall": recall_score(y, pred), f"{s} flagged rows": int(pred.sum())})
alt_rows.append(r)
alt_tab = pd.DataFrame(alt_rows).set_index("model")
print(alt_tab.T.round(4).to_string())
log_table(alt_tab, "alternative_models")
for name in order:
    key = name.split(" (")[0].replace(" ", "_")
    for s in ("val", "test"):
        mlflow.log_metric(f"alt_{key}_{s}_AP", alt_tab.loc[name, f"{s} AP"])

# SIM-level bootstrap of the test AP gap (XGBoost minus each model), 1,000 resamples of the 150 test SIMs.
te_sims = Y["test"]["sim_id"].values
uniq = np.unique(te_sims)
pos = {s: np.where(te_sims == s)[0] for s in uniq}
brng = np.random.default_rng(0)
boot = {n: [] for n in order if n != "XGBoost (final)"}
for _ in range(1000):
    idx = np.concatenate([pos[s] for s in brng.choice(uniq, len(uniq), replace=True)])
    if yte[idx].sum() == 0:
        continue
    ap_x = average_precision_score(yte[idx], scores["XGBoost (final)"]["test"][idx])
    for n in boot:
        boot[n].append(ap_x - average_precision_score(yte[idx], scores[n]["test"][idx]))
boot_tab = pd.DataFrame({n: {"test AP gap (XGBoost minus model)": np.mean(v), "2.5%": np.quantile(v, 0.025),
                             "97.5%": np.quantile(v, 0.975), "share of resamples XGBoost ahead": np.mean(np.array(v) > 0)}
                         for n, v in boot.items()}).T
print(); print(boot_tab.round(4).to_string())
log_table(boot_tab, "alternative_models_bootstrap")
TIMES["4 evaluate alternatives"] = time.perf_counter() - t0

model                      Isolation Forest alone Logistic regression       Random forest HistGradientBoosting  XGBoost (final) XGBoost (final), production gate p > 0.90
choice made on validation  saved model, no labels              C=0.01  min_samples_leaf=5       490 iterations  saved, 57 trees                                fixed 0.90
threshold (from val)                     0.090408            0.989104             0.40836             0.266477         0.773781                                       0.9
val AP                                   0.463265            0.808258            0.929306             0.922512         0.925353                                       NaN
val ROC AUC                              0.893307             0.94533            0.961618             0.951914         0.962003                                       NaN
val recall at 5% FPR                     0.583969            0.883588            0.929389             0.919847         0.927481                       


                        test AP gap (XGBoost minus model)    2.5%   97.5%  share of resamples XGBoost ahead
Isolation Forest alone                             0.4483  0.2236  0.7366                             1.000
Logistic regression                                0.1243  0.0646  0.2354                             1.000
Random forest                                     -0.0034 -0.0090  0.0009                             0.067
HistGradientBoosting                               0.0013 -0.0058  0.0075                             0.660


### 4. Conclusion

XGBoost is clearly better than the simpler options: logistic regression reaches test AP 0.81 and
Isolation Forest alone 0.49, against 0.93, and the SIM bootstrap puts XGBoost ahead in every resample.
It is not better than the other tree ensembles: random forest (test AP 0.933) and HistGradientBoosting
(0.928) are within the bootstrap noise of XGBoost (0.930; the 95% intervals of the gaps include 0). At
each model's own validation threshold for 0.95 precision, the three tree models also give similar
test flag precision and recall. So the choice of XGBoost is justified over linear and unsupervised
models on accuracy, and over the other tree ensembles only on practical grounds: it is already
deployed, it is small (57 trees), early stopping sets its size on validation, and its built-in
`pred_contribs` gives exact SHAP reasons for the API. Random forest would be an equally valid choice.
Two limits of this comparison: the logistic regression's best C (0.01) is at the edge of its grid, and
HistGradientBoosting used 490 of its 500 iterations, so both might gain slightly from a wider budget.

In [15]:
times = pd.Series(TIMES).round(1).rename("seconds")
print(times.to_string())
log_table(times.to_frame(), "part_times")
mlflow.log_metric("total_seconds", float(times.sum()))
print("MLflow run:", run.info.run_id)
mlflow.end_run()

0 data                                0.6
1 split check                         0.1
2a label shuffle                      1.7
2a label shuffle (early stopping)     2.2
2b temporal                           7.5
2c columns                            6.9
2d fit on train                       3.8
3 scoring                             1.5
3a false positives                    0.0
3b false negatives                    0.1
3b episode day 0                      0.3
4 fit alternatives                   41.6
4 evaluate alternatives              11.0
MLflow run: 669595fe07084312b755ecb8a89366bf
